# V101 - Feature-subspace bagging

Pre-registered plan: v98 bags users, but feature noise may dominate just as much with 319 sparse dims. Each bag draws bootstrap users and a random half of TF-IDF dims plus all assessment dims, refits Ridge, and averages out-of-bag. Gate against user-only bagging under the same grouped folds with a fresh seed and the usual locks. If feature noise matters, double randomization wins.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2626
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
FEAT_FRAC = 0.5
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v101")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Double randomization vs user bagging

Same folds judge both. Feature masks redraw per bag. Assessment dims stay intact as the stable core.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
P = W.shape[1]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)
W = W

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_oof(feat_mask, seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        cols = r.choice(P, int(P * FEAT_FRAC), replace=False) if feat_mask else np.arange(P)
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(W[idx][:, cols], Y[idx, j])
            acc[oob, j] += m.predict(W[oob][:, cols])
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

oof_u = bagged_oof(False, SEED)
oof_d = bagged_oof(True, SEED + 1)
strata = Y.argmax(1)
folds = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(W, strata, fold_groups))
fid = np.zeros(n_train, int)
for fi, (tr, va) in enumerate(folds):
    fid[va] = fi + 1
base = ndcg5(Y, oof_u)
d = ndcg5(Y, oof_d) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"user-bag={base.mean():.5f} double={ndcg5(Y, oof_d).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}]")
base = ndcg5(Y, oof_u)
wf = min(ndcg5(Y[fid == k], oof_d[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"worst-fold={wf:+.5f}")
print("ADOPT double randomization" if (lo > GATE_FLOOR and wf >= 0) else "KEEP user-only bagging; feature noise is not the bottleneck")

user-bag=0.63740 double=0.62292 delta=-0.01448 CI95=[-0.01796,-0.01083]
worst-fold=-0.01787
KEEP user-only bagging; feature noise is not the bottleneck
